# Highbay schema/prose fine-tune - v5

Objective: Fine-tune Qwen2.5-1.5B-Instruct for Typed Markdown IR & AST Generation (v5).

In [1]:
# 1. Mount Google Drive & Create Missing Training Directories
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/HighbayGeniusTraining"
TRAINING_OUTPUT_DIR = os.path.join(DRIVE_BASE, "training", "outputs")

os.makedirs(TRAINING_OUTPUT_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
# 2. Setup & Installation
!pip install -q torch transformers datasets unsloth trl

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 136.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3

In [3]:
# 3. Configuration Parameters
BASE_MODEL   = "unsloth/Qwen2.5-1.5B-Instruct"
RUN_ID       = "v5-qwen2.5-1.5b"
SEED         = 3407
EPOCHS       = 3
LR           = 2e-4
LORA_R       = 16
EVAL_FRACTION = 0.2
TARGET       = "ast"
MAX_SEQ_LENGTH = 2048

In [4]:
# 4. Model & LoRA Initialization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_R,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
# 5. Dataset Loading & Formatting
LOCAL_DATASET_PATH = "/content/TrainingExperiments/Highbay/Local/v5/inputs/synthetic_typed_markdown_v5.jsonl"
DRIVE_DATASET_PATH = os.path.join(DRIVE_BASE, "datasets", "processed", "synthetic_typed_markdown_v5.jsonl")

if os.path.exists(LOCAL_DATASET_PATH):
    DATASET_PATH = LOCAL_DATASET_PATH
    print(f"Reading dataset from local repository: {DATASET_PATH}")
elif os.path.exists(DRIVE_DATASET_PATH):
    DATASET_PATH = DRIVE_DATASET_PATH
    print(f"Reading dataset from Google Drive: {DATASET_PATH}")
else:
    raise FileNotFoundError(f"Could not find dataset at local path '{LOCAL_DATASET_PATH}' or Drive path '{DRIVE_DATASET_PATH}'")

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.train_test_split(test_size=EVAL_FRACTION, seed=SEED)

PROMPT_TEMPLATE = """Below is an instruction that describes a UI design action or natural language prompt. Write the corresponding Typed Markdown Intermediate Representation (IR).

### Instruction:
{prompt}

### Typed Markdown IR:
{target_ir}"""

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    targets = examples["target_ir"]
    texts = []
    for p, t in zip(prompts, targets):
        text = PROMPT_TEMPLATE.format(prompt=p, target_ir=t) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

Reading dataset from local repository: /content/TrainingExperiments/Highbay/Local/v5/inputs/synthetic_typed_markdown_v5.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [7]:
# 6. Training Arguments & Trainer Execution
output_run_dir = os.path.join(TRAINING_OUTPUT_DIR, f"outputs_{RUN_ID}")
os.makedirs(output_run_dir, exist_ok=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = output_run_dir,
        seed = SEED,
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/80 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/20 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.706482
2,3.557482
3,3.437647
4,3.623042
5,3.159230
6,3.142079
7,2.940210
8,2.738848
9,2.632545
10,2.043067


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/HighbayGeniusTraining/training/outputs/outputs_v5-qwen2.5-1.5b/checkpoint-30/tokenizer_config.json.


In [8]:
# 7. Save Adapter directly to Google Drive
adapter_save_path = os.path.join(TRAINING_OUTPUT_DIR, f"adapter_{RUN_ID}")
model.save_pretrained_merged(adapter_save_path, tokenizer, save_method="merged_16bit")
print(f"Adapter successfully saved to {adapter_save_path}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/HighbayGeniusTraining/training/outputs/adapter_v5-qwen2.5-1.5b/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:23<00:00, 23.41s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:07<00:00, 67.79s/it]

Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/HighbayGeniusTraining/training/outputs/adapter_v5-qwen2.5-1.5b`
Adapter successfully saved to /content/drive/MyDrive/HighbayGeniusTraining/training/outputs/adapter_v5-qwen2.5-1.5b


In [10]:
lora_adapter_save_path = os.path.join(TRAINING_OUTPUT_DIR, f"lora_adapter_{RUN_ID}")
model.save_pretrained(lora_adapter_save_path)
print(f"LoRA adapter successfully saved to {lora_adapter_save_path}")

LoRA adapter successfully saved to /content/drive/MyDrive/HighbayGeniusTraining/training/outputs/lora_adapter_v5-qwen2.5-1.5b


In [9]:
gguf_output_path = os.path.join(TRAINING_OUTPUT_DIR, f"gguf_model_{RUN_ID}")
model.save_pretrained_gguf(gguf_output_path, tokenizer, quantization_method = "q4_k_m")
print(f"GGUF model successfully saved to {gguf_output_path}")

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/HighbayGeniusTraining/training/outputs/gguf_model_v5-qwen2.5-1.5b/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:49<00:00, 49.69s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:52<00:00, 52.02s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/HighbayGeniusTraining/training/outputs/gguf_model_v5-qwen2.5-1.5b`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10360-mix-87da1a2 (app-b10360-mix-87da1a2-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/HighbayGeniusTraining/training/outputs/gguf_model_v5-qwen2.5-1.5b_gguf/qwen2.5-1.5b-instruct.F16.gguf']
Unsloth: 

### 8. Generate Standalone GGUF LoRA Adapter for `llama.cpp`

To address the need for a standalone GGUF LoRA adapter (which can be applied to a GGUF base model using `llama.cpp`'s `--lora` flag), we'll follow these steps:

1.  **Save LoRA adapter locally:** This will save only the adapter weights and configuration.
2.  **Clone `llama.cpp`:** We need the `convert_lora_to_gguf.py` script from this repository.
3.  **Convert LoRA to GGUF:** The script will take the original `BASE_MODEL` ID and your saved local LoRA adapter to create a GGUF formatted adapter file.
4.  **Copy to Drive:** The local adapter and the new GGUF adapter will be copied to your Google Drive.

In [11]:
# 8.1 Save the LoRA adapter locally
LOCAL_ADAPTER_DIR = os.path.join("/content", f"outputs_{RUN_ID}", "adapter")
os.makedirs(LOCAL_ADAPTER_DIR, exist_ok=True)

# Save only the LoRA adapter (not merged)
model.save_pretrained(LOCAL_ADAPTER_DIR)
print(f"LoRA adapter successfully saved locally to {LOCAL_ADAPTER_DIR}")

LoRA adapter successfully saved locally to /content/outputs_v5-qwen2.5-1.5b/adapter


In [20]:
# 8.2 Clone llama.cpp to get the conversion script and its dependencies
import os

# Remove existing llama.cpp directory if it exists to ensure a clean clone
if os.path.exists('/content/llama.cpp'):
    !rm -rf /content/llama.cpp

!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /content/llama.cpp

# Install gguf library if needed by the conversion script
!pip install -q gguf

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 3774, done.
remote: Counting objects: 100% (3774/3774), done.
remote: Compressing objects: 100% (3067/3067), done.
remote: Total 3774 (delta 663), reused 2683 (delta 623), pack-reused 0 (from 0)
Receiving objects: 100% (3774/3774), 35.35 MiB | 1.64 MiB/s, done.
Resolving deltas: 100% (663/663), done.
Updating files: 100% (3425/3425), done.


In [23]:
# 8.3 Run convert_lora_to_gguf.py
LOCAL_GGUF_LORA_ADAPTER_PATH = os.path.join("/content", f"lora_adapter_{RUN_ID}.gguf")

print(f"Converting LoRA adapter to GGUF format using base model: {BASE_MODEL}")
!python /content/llama.cpp/convert_lora_to_gguf.py \
    --base-model-id {BASE_MODEL} \
    --outfile {LOCAL_GGUF_LORA_ADAPTER_PATH} \
    {LOCAL_ADAPTER_DIR}

print(f"Standalone GGUF LoRA adapter saved to {LOCAL_GGUF_LORA_ADAPTER_PATH}")

Converting LoRA adapter to GGUF format using base model: unsloth/Qwen2.5-1.5B-Instruct
/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
INFO:lora-to-gguf:Loading base model from Hugging Face: unsloth/Qwen2.5-1.5B-Instruct
INFO:httpx:HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-1.5B-Instruct/b2e27ed8774d78eb2ee474cfe99d2d3b5fae11e5/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/model

In [24]:
# 8.4 Copy local adapter folder and generated GGUF LoRA adapter to Google Drive
DRIVE_TARGET_DIR = os.path.join(TRAINING_OUTPUT_DIR, f"adapter_{RUN_ID}")
os.makedirs(DRIVE_TARGET_DIR, exist_ok=True)

# Copy the local adapter folder contents
!cp -r {LOCAL_ADAPTER_DIR}/* {DRIVE_TARGET_DIR}/
print(f"LoRA adapter folder copied to Google Drive: {DRIVE_TARGET_DIR}")

# Copy the generated GGUF LoRA adapter file
!cp {LOCAL_GGUF_LORA_ADAPTER_PATH} {DRIVE_TARGET_DIR}/
print(f"Standalone GGUF LoRA adapter copied to Google Drive: {os.path.join(DRIVE_TARGET_DIR, os.path.basename(LOCAL_GGUF_LORA_ADAPTER_PATH))}")

LoRA adapter folder copied to Google Drive: /content/drive/MyDrive/HighbayGeniusTraining/training/outputs/adapter_v5-qwen2.5-1.5b
Standalone GGUF LoRA adapter copied to Google Drive: /content/drive/MyDrive/HighbayGeniusTraining/training/outputs/adapter_v5-qwen2.5-1.5b/lora_adapter_v5-qwen2.5-1.5b.gguf
